In [ ]:
import sys
sys.path.append("../../..")

In [ ]:
import os
import pathlib
import shutil

import analytiq_data as ad
import asyncio

DOCROUTER_TMP = pathlib.Path("/tmp/docrouter")
if DOCROUTER_TMP.exists():
    shutil.rmtree(DOCROUTER_TMP)
DOCROUTER_TMP.mkdir(parents=True, exist_ok=True)

In [ ]:
ad.common.setup()

In [ ]:
BMDS_BASE_URL = "http://10.83.8.98:8000"
BMDS_WORKSPACE_TOKEN = ""
BMDS_ORG_ID = "68be2d153ab854a43514f12d"
DOCUMENT_LIMIT = 1000

In [ ]:
import json
import requests

def list_workspace_documents_rest(
    base_url: str,
    api_token: str,
    organization_id: str,
    *,
    tag_ids: list[str] | None = None,
    name_search: str | None = None,
    metadata_search: dict[str, str] | None = None,
    max_documents: int = 1000,
) -> list[dict]:
    """
    List documents in a DocRouter workspace via the REST API (up to max_documents).
    Results are ordered by upload timestamp, most recent first.
    """
    url = f"{base_url.rstrip('/')}/v0/orgs/{organization_id}/documents"
    headers = {"Authorization": f"Bearer {api_token}"}
    sort = json.dumps([{"field": "upload_date", "sort": "desc"}])

    all_documents: list[dict] = []
    skip = 0
    page_limit = 100  # API maximum per page

    while len(all_documents) < max_documents:
        params: dict[str, str | int] = {
            "skip": skip,
            "limit": min(page_limit, max_documents - len(all_documents)),
            "sort": sort,
        }
        if tag_ids:
            params["tag_ids"] = ",".join(tag_ids)
        if name_search:
            params["name_search"] = name_search
        if metadata_search:
            from urllib.parse import quote

            params["metadata_search"] = ",".join(
                f"{quote(str(k))}={quote(str(v))}" for k, v in metadata_search.items()
            )

        response = requests.get(url, headers=headers, params=params, timeout=120)
        response.raise_for_status()
        batch = response.json().get("documents", [])
        all_documents.extend(batch)

        if len(batch) < params["limit"]:
            break
        skip += params["limit"]

    return all_documents[:max_documents]





In [ ]:
# List the first 1000 documents in the workspace (newest first)
workspace_documents = list_workspace_documents_rest(
    BMDS_BASE_URL,
    BMDS_WORKSPACE_TOKEN,
    BMDS_ORG_ID,
    max_documents=1000,
)

print(f"Found {len(workspace_documents)} workspace documents (newest first, max 1000):\n")
for i, doc in enumerate(workspace_documents, 1):
    print(f"{i}. {doc.get('document_name', '(unnamed)')}")
    print(f"   ID: {doc.get('id')}")
    print(f"   Uploaded: {doc.get('upload_date')}")
    print(f"   State: {doc.get('state')}")
    if doc.get("tag_ids"):
        print(f"   Tag IDs: {doc.get('tag_ids')}")
    if doc.get("metadata"):
        print(f"   Metadata: {doc.get('metadata')}")
    print()

In [ ]:
import base64
import gzip


def download_document_ocr(
    base_url: str,
    api_token: str,
    organization_id: str,
    document_id: str,
    *,
    output_dir: pathlib.Path = DOCROUTER_TMP,
    compressed: bool = False,
) -> pathlib.Path:
    """
    Download OCR JSON for a document via the REST API and save to
    /tmp/docrouter/<document_id>_ocr.json (or output_dir).

    compressed=False: pretty-printed JSON (format=plain).
    compressed=True: gzip-compressed payload encrypted with ad.crypto.encrypt_secret
    (format=gzip over the wire; file holds v2:... ciphertext).
    """
    url = (
        f"{base_url.rstrip('/')}/v0/orgs/{organization_id}"
        f"/ocr/download/json/{document_id}"
    )
    headers = {"Authorization": f"Bearer {api_token}"}
    out_path = output_dir / f"{document_id}_ocr.json"

    if not compressed:
        response = requests.get(
            url, headers=headers, params={"format": "plain"}, timeout=120
        )
        response.raise_for_status()
        out_path.write_text(
            json.dumps(response.json(), indent=2),
            encoding="utf-8",
        )
        return out_path

    response = requests.get(
        url,
        headers=headers,
        params={"format": "gzip"},
        stream=True,
        timeout=120,
    )
    response.raise_for_status()
    response.raw.decode_content = False
    gzip_bytes = response.raw.read()
    encrypted = ad.crypto.encrypt_secret(
        base64.b64encode(gzip_bytes).decode("ascii")
    )
    out_path.write_text(encrypted, encoding="utf-8")
    return out_path


In [ ]:
import time

download_times: list[float] = []
failed_ids: list[str] = []

n_docs = len(workspace_documents)
print(f"Downloading OCR JSON for {n_docs} documents (compressed=True)...")

for i, doc in enumerate(workspace_documents, 1):
    doc_id = doc["id"]
    t0 = time.perf_counter()
    try:
        download_document_ocr(
            BMDS_BASE_URL,
            BMDS_WORKSPACE_TOKEN,
            BMDS_ORG_ID,
            doc_id,
            compressed=True,
        )
        elapsed = time.perf_counter() - t0
        download_times.append(elapsed)
        print(f"{i}: Downloaded {doc_id} in {elapsed:.3f}s")
    except Exception as e:
        failed_ids.append(doc_id)
        print(f"  [{i}/{n_docs}] {doc_id} FAILED: {e}")
        continue

    if i % 50 == 0 or i == n_docs:
        print(f"  [{i}/{n_docs}] last download {elapsed:.3f}s")

if download_times:
    print(f"\nDownload time (seconds) over {len(download_times)} successful downloads:")
    print(f"  min: {min(download_times):.3f}")
    print(f"  max: {max(download_times):.3f}")
    print(f"  avg: {sum(download_times) / len(download_times):.3f}")
    print(f"  total: {sum(download_times):.1f}")
if failed_ids:
    print(f"  failed: {len(failed_ids)}")


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

N_THREADS = 16

def _download_ocr_shard(shard_idx: int) -> tuple[list[float], list[str]]:
    """Thread shard_idx downloads documents at indices shard_idx, shard_idx+N_THREADS, ..."""
    times: list[float] = []
    failed: list[str] = []
    shard_docs = workspace_documents[shard_idx::N_THREADS]

    for doc in shard_docs:
        doc_id = doc["id"]
        t0 = time.perf_counter()
        try:
            download_document_ocr(
                BMDS_BASE_URL,
                BMDS_WORKSPACE_TOKEN,
                BMDS_ORG_ID,
                doc_id,
                compressed=True,
            )
            times.append(time.perf_counter() - t0)
            print(f"  thread {shard_idx} {doc_id} OK: {times[-1]:.3f}s")
        except Exception as e:
            failed.append(doc_id)
            print(f"  thread {shard_idx} {doc_id} FAILED: {e}")

    print(f"  thread {shard_idx} done: {len(times)} ok, {len(failed)} failed")
    return times, failed


n_docs = len(workspace_documents)
print(
    f"Downloading OCR for {n_docs} documents "
    f"(compressed=True, {N_THREADS} threads, stride={N_THREADS})..."
)
wall_t0 = time.perf_counter()

download_times_parallel: list[float] = []
failed_ids_parallel: list[str] = []

with ThreadPoolExecutor(max_workers=N_THREADS) as executor:
    futures = [
        executor.submit(_download_ocr_shard, shard_idx)
        for shard_idx in range(N_THREADS)
    ]
    for fut in as_completed(futures):
        times, failed = fut.result()
        download_times_parallel.extend(times)
        failed_ids_parallel.extend(failed)

wall_elapsed = time.perf_counter() - wall_t0
print(f"\nWall time: {wall_elapsed:.1f}s")
if download_times_parallel:
    print(
        f"Per-download time (seconds) over {len(download_times_parallel)} successful:"
    )
    print(f"  min: {min(download_times_parallel):.3f}")
    print(f"  max: {max(download_times_parallel):.3f}")
    print(f"  avg: {sum(download_times_parallel) / len(download_times_parallel):.3f}")
    print(f"  sum: {sum(download_times_parallel):.1f}")
if failed_ids_parallel:
    print(f"  failed: {len(failed_ids_parallel)}")
print(f"  downloaded: {len(download_times_parallel)} / {n_docs}")
